In [6]:
from jupyter_dash import JupyterDash

import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import base64
import pandas as pd

from animal_shelter import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()

username = "aacuser"
password = "SNHU1234"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))

if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)

app = JupyterDash(__name__)

image_filename = "Grazioso Salvare Logo.png"
encoded_image = base64.b64encode(open(image_filename, "rb").read())

app.layout = html.Div([
    html.Center([
        html.Img(
            src="data:image/png;base64,{}".format(encoded_image.decode()),
            style={"height": "120px"}
        ),
        html.H1("Grazioso Salvare Animal Rescue Dashboard"),
        html.H3("Matthew Rios - CS-340 Project Two")
    ]),

    html.Hr(),

    html.Div([
        html.H4("Select Rescue Type Filter:"),

        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster or Individual Tracking", "value": "disaster"},
                {"label": "Reset", "value": "reset"}
            ],
            value="reset",
            labelStyle={"display": "inline-block", "margin-right": "20px"}
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict("records"),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "150px",
            "maxWidth": "200px",
            "whiteSpace": "normal"
        },
        style_header={
            "fontWeight": "bold"
        }
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={"width": "50%"}
            ),
            html.Div(
                id="map-id",
                className="col s12 m6",
                style={"width": "50%"}
            )
        ]
    )
])


@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    if filter_type == "water":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland Mix"
                ]
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "mountain":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == "disaster":
        query = {
            "animal_type": "Dog",
            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    return dff.to_dict("records")


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or "breed" not in dff.columns:
        return []

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names="breed",
                title="Preferred Rescue Dog Breeds"
            )
        )
    ]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):

    if selected_columns is None:
        selected_columns = []

    return [
        {
            "if": {"column_id": col},
            "background_color": "#D2F3FF"
        }
        for col in selected_columns
    ]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):

    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return []

    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    try:
        lat = float(dff.iloc[row, 13])
        lon = float(dff.iloc[row, 14])
    except:
        return []

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(dff.iloc[row, 4])),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(dff.iloc[row, 9]))
                        ])
                    ]
                )
            ]
        )
    ]


app.run_server()

Dash app running on https://methodlunch-novelinvite-3000.codio.io/proxy/8050/
